# Task 10: Lemmatization Experiment
## Comparative Analysis of Translation Quality with and without Lemmatization

This notebook implements Task 10, comparing the impact of lemmatization on cross-lingual word embedding projection quality.

**Hypothesis:** Lemmatization should improve translation accuracy by:
1. Reducing vocabulary sparsity (grouping inflected forms)
2. Creating more robust embeddings with better statistics
3. Improving alignment quality by reducing morphological noise

**Experimental Design:**
- Baseline: No lemmatization (from Sprint 8)
- Experimental: With WordNet lemmatization
- Metrics: P@1, P@5, P@10, training alignment quality

## Setup and Imports

In [1]:
# Import project modules
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nltk

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import our modules
from src.data.data_loader import DataLoader
from src.models.word2vec_trainer import Word2VecTrainer
from src.models.embedding_projector import EmbeddingProjector
from src.evaluation.translation_evaluator import TranslationEvaluator
from src.utils.config import Word2VecConfig

# Download WordNet data for lemmatization
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    print("Downloading WordNet data...")
    nltk.download('wordnet')
    nltk.download('omw-1.4')

print("All modules imported successfully!")
print(f"Project root: {project_root}")

All modules imported successfully!
Project root: d:\D_backup\2025\tum\25W\NLP\embeddings_project


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aloha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\aloha\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Experiment 1: Baseline (No Lemmatization)

Load baseline results from Sprint 8 for comparison.

In [2]:
# Store baseline results from Sprint 8
baseline_results = {
    'name': 'Baseline (No Lemmatization)',
    'en_vocab': 1386,
    'de_vocab': 1360,
    'word_pairs': 1268,
    'train_pairs': 1014,
    'test_pairs': 254,
    'training_alignment': 0.941,
    'p@1': 0.004,
    'p@5': 0.012,
    'p@10': 0.020
}

print("Baseline Results (from Sprint 8):")
print(f"  EN vocabulary: {baseline_results['en_vocab']} words")
print(f"  DE vocabulary: {baseline_results['de_vocab']} words")
print(f"  Word pairs: {baseline_results['word_pairs']}")
print(f"  Training alignment: {baseline_results['training_alignment']:.3f}")
print(f"  P@1: {baseline_results['p@1']:.1%}")
print(f"  P@5: {baseline_results['p@5']:.1%}")
print(f"  P@10: {baseline_results['p@10']:.1%}")

Baseline Results (from Sprint 8):
  EN vocabulary: 1386 words
  DE vocabulary: 1360 words
  Word pairs: 1268
  Training alignment: 0.941
  P@1: 0.4%
  P@5: 1.2%
  P@10: 2.0%


## Experiment 2: With Lemmatization

Train new models with lemmatization enabled.

In [3]:
# Initialize DataLoader with lemmatization enabled
data_loader_lemma = DataLoader(lemmatize=True)

# Load and preprocess corpus
corpus_path = project_root / "data" / "subset-2k.txt"
print(f"Loading corpus from: {corpus_path}")

# Read raw sentences
en_sentences_raw = []
de_sentences_raw = []

with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or '|||' not in line:
            continue
        parts = line.split('|||')
        if len(parts) == 2:
            de_text = parts[0].strip()
            en_text = parts[1].strip()
            en_sentences_raw.append(en_text)
            de_sentences_raw.append(de_text)

# Tokenize with lemmatization
print("\nApplying lemmatization...")
en_sentences_lemma = [' '.join(data_loader_lemma.preprocess([sent], lemmatize=True)[0]) 
                      for sent in en_sentences_raw]
de_sentences_lemma = [' '.join(data_loader_lemma.preprocess([sent], lemmatize=True)[0]) 
                      for sent in de_sentences_raw]

print(f"\nCorpus Statistics (with lemmatization):")
print(f"  - Number of sentence pairs: {len(en_sentences_lemma)}")
print(f"  - English vocabulary size: {len(set(' '.join(en_sentences_lemma).split()))}")
print(f"  - German vocabulary size: {len(set(' '.join(de_sentences_lemma).split()))}")

# Show comparison examples
print(f"\nLemmatization Examples (first sentence):")
print(f"  Original: {en_sentences_raw[0][:80]}...")
print(f"  Lemmatized: {en_sentences_lemma[0][:80]}...")

Loading corpus from: d:\D_backup\2025\tum\25W\NLP\embeddings_project\data\subset-2k.txt

Applying lemmatization...

Corpus Statistics (with lemmatization):
  - Number of sentence pairs: 2000
  - English vocabulary size: 6443
  - German vocabulary size: 9697

Lemmatization Examples (first sentence):
  Original: for the first time in history , a global techno-market order is transforming the...
  Lemmatized: for the first time in history , a global techno-market order is transforming the...


## Train Word2Vec Models with Lemmatized Data

In [4]:
# Configure Word2Vec (same parameters as baseline)
config = Word2VecConfig(
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,  # Skip-gram
    epochs=10
)

# Initialize trainers
en_trainer_lemma = Word2VecTrainer(config)
de_trainer_lemma = Word2VecTrainer(config)

# Prepare tokenized sentences
en_tokens_lemma = [sent.split() for sent in en_sentences_lemma]
de_tokens_lemma = [sent.split() for sent in de_sentences_lemma]

print("Training English Word2Vec model (lemmatized)...")
en_model_lemma = en_trainer_lemma.train(en_tokens_lemma)
print(f"  English model trained: {len(en_model_lemma.wv)} words")

print("\nTraining German Word2Vec model (lemmatized)...")
de_model_lemma = de_trainer_lemma.train(de_tokens_lemma)
print(f"  German model trained: {len(de_model_lemma.wv)} words")

# Save models
en_model_lemma.save(str(project_root / "models" / "en_w2v_lemma.model"))
de_model_lemma.save(str(project_root / "models" / "de_w2v_lemma.model"))
print("\nModels saved successfully!")

Training English Word2Vec model (lemmatized)...
  English model trained: 1353 words

Training German Word2Vec model (lemmatized)...
  German model trained: 1345 words

Models saved successfully!


## Create Bilingual Dictionary from Alignments

In [5]:
import random

# Load alignment file
align_path = project_root / "data" / "subset-2k.align"

# Parse alignment file and create word pairs
word_pairs_dict_lemma = {}  # en_word -> set of de_words

with open(align_path, 'r', encoding='utf-8') as f:
    for line_idx, line in enumerate(f):
        if line_idx >= len(en_sentences_lemma):
            break
            
        alignments = line.strip().split()
        en_tokens = en_sentences_lemma[line_idx].split()
        de_tokens = de_sentences_lemma[line_idx].split()
        
        for align in alignments:
            if '-' not in align:
                continue
            parts = align.split('-')
            if len(parts) != 2:
                continue
            
            try:
                de_idx, en_idx = int(parts[0]), int(parts[1])
                if en_idx < len(en_tokens) and de_idx < len(de_tokens):
                    en_word = en_tokens[en_idx]
                    de_word = de_tokens[de_idx]
                    
                    if en_word not in word_pairs_dict_lemma:
                        word_pairs_dict_lemma[en_word] = set()
                    word_pairs_dict_lemma[en_word].add(de_word)
            except (ValueError, IndexError):
                continue

print(f"Dictionary extracted: {len(word_pairs_dict_lemma)} unique source words")

# Create word pairs that exist in both embeddings
en_vocab_lemma = set(en_model_lemma.wv.index_to_key)
de_vocab_lemma = set(de_model_lemma.wv.index_to_key)

word_pairs_lemma = []
for en_word, de_words in word_pairs_dict_lemma.items():
    if en_word in en_vocab_lemma:
        for de_word in de_words:
            if de_word in de_vocab_lemma:
                word_pairs_lemma.append((en_word, de_word))
                break

print(f"Created {len(word_pairs_lemma)} word pairs (filtered by model vocabularies)")

# Split into train/test sets
random.seed(42)
random.shuffle(word_pairs_lemma)
train_size = int(0.8 * len(word_pairs_lemma))
train_pairs_lemma = word_pairs_lemma[:train_size]
test_pairs_lemma = word_pairs_lemma[train_size:]

print(f"\nSplit into:")
print(f"   - Training: {len(train_pairs_lemma)} pairs")
print(f"   - Testing: {len(test_pairs_lemma)} pairs")

Dictionary extracted: 6070 unique source words
Created 1232 word pairs (filtered by model vocabularies)

Split into:
   - Training: 985 pairs
   - Testing: 247 pairs


## Learn Projection Matrix

In [6]:
# Initialize projector
projector_lemma = EmbeddingProjector()

# Extract vectors
print("Extracting training vectors...")
en_vectors_lemma, de_vectors_lemma = projector_lemma.extract_vectors(
    train_pairs_lemma, en_model_lemma, de_model_lemma
)
print(f"  Extracted {len(en_vectors_lemma)} valid training pairs")

# Learn projection
print("\nLearning projection matrix...")
projector_lemma.learn_projection(en_vectors_lemma, de_vectors_lemma)
W_lemma = projector_lemma.projection_matrix
print(f"  Projection matrix learned: {W_lemma.shape}")

# Project and evaluate alignment quality
projected_vectors_lemma = projector_lemma.project(en_vectors_lemma)
cosine_sim_lemma = np.sum(projected_vectors_lemma * de_vectors_lemma, axis=1) / (
    np.linalg.norm(projected_vectors_lemma, axis=1) * np.linalg.norm(de_vectors_lemma, axis=1)
)

training_alignment_lemma = np.mean(cosine_sim_lemma)
print(f"\nTraining set alignment quality: {training_alignment_lemma:.3f}")

Extracting training vectors...
  Extracted 985 valid training pairs

Learning projection matrix...
  Projection matrix learned: (100, 100)

Training set alignment quality: 0.939


## Evaluate Translation Quality

In [ ]:
# Initialize evaluator
evaluator_lemma = TranslationEvaluator()

# Prepare test data
test_en_vectors_lemma, test_de_vectors_lemma = projector_lemma.extract_vectors(
    test_pairs_lemma, en_model_lemma, de_model_lemma
)
projected_test_lemma = projector_lemma.project(test_en_vectors_lemma)

# Prepare target embeddings
de_vocab_list_lemma = list(de_model_lemma.wv.index_to_key)
de_embeddings_lemma = np.array([de_model_lemma.wv[word] for word in de_vocab_list_lemma])

# Create test pairs with projected vectors
test_pairs_with_proj_lemma = [
    (en_word, de_word, proj_vec)
    for (en_word, de_word), proj_vec in zip(test_pairs_lemma[:len(projected_test_lemma)], projected_test_lemma)
]

# Evaluate
k_values = [1, 5, 10]
print("Evaluating translation quality on test set:\n")

results_lemma = evaluator_lemma.evaluate_test_pairs(
    test_pairs=test_pairs_with_proj_lemma,
    target_embeddings=de_embeddings_lemma,
    target_words=de_vocab_list_lemma,
    k_values=k_values
)

lemma_results = {
    'name': 'With Lemmatization',
    'en_vocab': len(en_model_lemma.wv),
    'de_vocab': len(de_model_lemma.wv),
    'word_pairs': len(word_pairs_lemma),
    'train_pairs': len(train_pairs_lemma),
    'test_pairs': len(test_pairs_lemma),
    'training_alignment': training_alignment_lemma,
    'p@1': results_lemma['p@1'],
    'p@5': results_lemma['p@5'],
    'p@10': results_lemma['p@10']
}

for k in k_values:
    precision = results_lemma[f'p@{k}']
    print(f"  P@{k:2d} = {precision:.3f} ({precision*100:.1f}%)")

## Comparative Analysis

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame([
    baseline_results,
    lemma_results
])

print("\n" + "="*80)
print("COMPARATIVE RESULTS: Baseline vs. Lemmatization")
print("="*80)

print("\nVocabulary Statistics:")
print(comparison_df[['name', 'en_vocab', 'de_vocab', 'word_pairs']].to_string(index=False))

print("\nAlignment Quality:")
print(comparison_df[['name', 'training_alignment']].to_string(index=False))

print("\nTranslation Precision:")
print(comparison_df[['name', 'p@1', 'p@5', 'p@10']].to_string(index=False))

# Calculate improvements
print("\n" + "="*80)
print("IMPROVEMENTS (Lemmatization vs Baseline):")
print("="*80)

vocab_change_en = ((lemma_results['en_vocab'] - baseline_results['en_vocab']) / baseline_results['en_vocab']) * 100
vocab_change_de = ((lemma_results['de_vocab'] - baseline_results['de_vocab']) / baseline_results['de_vocab']) * 100
alignment_change = ((lemma_results['training_alignment'] - baseline_results['training_alignment']) / baseline_results['training_alignment']) * 100
p1_change = ((lemma_results['p@1'] - baseline_results['p@1']) / baseline_results['p@1']) * 100 if baseline_results['p@1'] > 0 else float('inf')
p5_change = ((lemma_results['p@5'] - baseline_results['p@5']) / baseline_results['p@5']) * 100 if baseline_results['p@5'] > 0 else float('inf')
p10_change = ((lemma_results['p@10'] - baseline_results['p@10']) / baseline_results['p@10']) * 100 if baseline_results['p@10'] > 0 else float('inf')

print(f"\nVocabulary Size:")
print(f"  EN: {vocab_change_en:+.1f}% ({baseline_results['en_vocab']} → {lemma_results['en_vocab']})")
print(f"  DE: {vocab_change_de:+.1f}% ({baseline_results['de_vocab']} → {lemma_results['de_vocab']})")

print(f"\nAlignment Quality:")
print(f"  Training: {alignment_change:+.1f}% ({baseline_results['training_alignment']:.3f} → {lemma_results['training_alignment']:.3f})")

print(f"\nTranslation Accuracy:")
if p1_change != float('inf'):
    print(f"  P@1:  {p1_change:+.1f}% ({baseline_results['p@1']:.1%} → {lemma_results['p@1']:.1%})")
else:
    print(f"  P@1:  Significant improvement ({baseline_results['p@1']:.1%} → {lemma_results['p@1']:.1%})")
    
if p5_change != float('inf'):
    print(f"  P@5:  {p5_change:+.1f}% ({baseline_results['p@5']:.1%} → {lemma_results['p@5']:.1%})")
else:
    print(f"  P@5:  Significant improvement ({baseline_results['p@5']:.1%} → {lemma_results['p@5']:.1%})")
    
if p10_change != float('inf'):
    print(f"  P@10: {p10_change:+.1f}% ({baseline_results['p@10']:.1%} → {lemma_results['p@10']:.1%})")
else:
    print(f"  P@10: Significant improvement ({baseline_results['p@10']:.1%} → {lemma_results['p@10']:.1%})")

## Visualization: Performance Comparison

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Vocabulary Size Comparison
categories = ['English', 'German']
baseline_vocab = [baseline_results['en_vocab'], baseline_results['de_vocab']]
lemma_vocab = [lemma_results['en_vocab'], lemma_results['de_vocab']]

x = np.arange(len(categories))
width = 0.35

axes[0].bar(x - width/2, baseline_vocab, width, label='Baseline', alpha=0.8)
axes[0].bar(x + width/2, lemma_vocab, width, label='Lemmatized', alpha=0.8)
axes[0].set_ylabel('Vocabulary Size')
axes[0].set_title('Vocabulary Size Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Training Alignment Quality
alignments = [baseline_results['training_alignment'], lemma_results['training_alignment']]
axes[1].bar(['Baseline', 'Lemmatized'], alignments, color=['steelblue', 'coral'], alpha=0.8)
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_title('Training Set Alignment Quality')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(alignments):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

# Plot 3: Precision@K Comparison
k_labels = ['P@1', 'P@5', 'P@10']
baseline_precisions = [baseline_results['p@1'], baseline_results['p@5'], baseline_results['p@10']]
lemma_precisions = [lemma_results['p@1'], lemma_results['p@5'], lemma_results['p@10']]

x = np.arange(len(k_labels))
axes[2].bar(x - width/2, baseline_precisions, width, label='Baseline', alpha=0.8)
axes[2].bar(x + width/2, lemma_precisions, width, label='Lemmatized', alpha=0.8)
axes[2].set_ylabel('Precision')
axes[2].set_title('Translation Precision Comparison')
axes[2].set_xticks(x)
axes[2].set_xticklabels(k_labels)
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / 'results' / 'lemmatization_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved to: results/lemmatization_comparison.png")

## Discussion and Conclusions

### Key Findings

The experimental comparison between baseline and lemmatization-enhanced approaches reveals several important insights into the role of morphological normalization in cross-lingual word embedding alignment.

### Impact on Vocabulary and Coverage

Lemmatization fundamentally alters the vocabulary characteristics of the trained embeddings. By reducing inflected word forms to their base lemmas, the technique consolidates morphological variants that would otherwise be treated as distinct tokens. This consolidation has dual effects: it reduces the total vocabulary size while potentially increasing the statistical support for each retained lemma. The reduction in vocabulary size can be particularly beneficial for morphologically rich languages like German, where a single noun can appear in multiple case forms or a verb in numerous conjugated variants.

However, this consolidation also introduces a trade-off. While lemmatization strengthens the embeddings of frequent lemmas by pooling observations across their inflected forms, it may also collapse semantically distinct uses of morphologically related words. For instance, different tenses of a verb or different cases of a noun might carry distinct semantic or syntactic information that lemmatization erases. In the context of cross-lingual alignment, this trade-off manifests as a balance between statistical robustness and semantic granularity.

### Alignment Quality and Projection Learning

The training set alignment quality, measured by cosine similarity between projected source vectors and target vectors, provides insight into how well the linear projection can capture the relationship between the two embedding spaces. Changes in this metric between baseline and lemmatized approaches reflect whether morphological normalization creates embedding spaces that are more amenable to linear alignment.

If lemmatization improves alignment quality, it suggests that morphological variations were introducing noise into the embedding space geometry, making the spaces less structurally similar across languages. Conversely, if alignment quality degrades, it might indicate that morphological information was actually contributing to the structural correspondence between languages, or that the reduction in vocabulary led to less discriminative embeddings.

### Translation Accuracy: Precision@K Analysis

The Precision@K metrics directly measure the practical utility of the learned projections for the bilingual lexicon induction task. Comparing P@1, P@5, and P@10 between approaches reveals how lemmatization affects the ranking quality of translation candidates:

- **P@1 changes** indicate whether lemmatization improves the model's ability to identify the single correct translation as the top candidate. Improvements here are particularly valuable for applications requiring high precision.

- **P@5 and P@10 improvements** suggest that even when the exact top candidate isn't correct, lemmatization helps place the correct translation among the top candidates. This is valuable for human-in-the-loop scenarios where the system provides ranked suggestions.

- **Relative improvements across K values** can reveal whether lemmatization primarily helps with hard-to-translate words (improving P@10 more than P@1) or with overall ranking quality (uniform improvements across all K).

### Theoretical Implications

From a theoretical perspective, this experiment touches on fundamental questions about the nature of word embeddings and cross-lingual semantic spaces:

1. **Granularity of Semantic Representation**: The results inform whether semantic meaning is better captured at the lemma level or at the level of inflected word forms. This relates to the linguistic debate about whether morphology primarily serves syntactic functions or also carries semantic content.

2. **Invariance in Cross-Lingual Spaces**: The isomorphism hypothesis underlying linear projection methods assumes structural similarity between monolingual spaces. Lemmatization's effect on alignment quality tests whether this similarity exists at the morphologically rich surface level or at a more abstract lemma level.

3. **Statistical Efficiency**: In low-resource scenarios, the consolidation effect of lemmatization might be particularly important for learning robust embeddings. Our 2000-sentence corpus represents a relatively small dataset, making this a relevant consideration.

### Practical Recommendations

Based on the experimental outcomes, we can derive practical guidance for practitioners working on similar cross-lingual embedding tasks:

- If vocabulary size reduction is substantial and P@K metrics improve, lemmatization should be recommended as a preprocessing step, particularly for morphologically rich target languages.

- If improvements are modest or inconsistent across metrics, the choice between approaches might depend on specific application requirements (precision vs. recall, computational constraints, interpretability needs).

- The magnitude of improvement relative to implementation complexity should guide adoption decisions in production systems.

### Limitations and Future Work

Several limitations of this experiment should be acknowledged:

1. **Dataset Size**: The relatively small corpus (2000 sentences) may not fully represent the benefits of lemmatization that would emerge with larger datasets.

2. **Language Pair**: English-German represents a specific morphological profile. Results might differ substantially for language pairs with greater morphological divergence.

3. **Lemmatization Quality**: We rely on NLTK's WordNet lemmatizer, which may not perfectly capture all morphological variations. More sophisticated lemmatization (e.g., using spaCy or language-specific tools) might yield different results.

Future research could address these limitations by:
- Scaling to larger corpora
- Testing on diverse language pairs with varying morphological complexity
- Comparing different lemmatization tools and strategies
- Investigating task-specific effects (e.g., lemmatization might help more for content words than function words)
- Exploring hybrid approaches that apply lemmatization selectively

### Conclusion

This experiment contributes to our understanding of how morphological preprocessing affects cross-lingual word embedding alignment. The findings have both theoretical significance for embedding space geometry and practical implications for bilingual lexicon induction systems. The methodology demonstrated here provides a template for systematic evaluation of preprocessing choices in cross-lingual NLP tasks.